In [ ]:
import re

import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline


RANDOM_STATE = 42

# 1) Load the clean dataset

In [ ]:
df = pd.read_csv("CLEAN_PATH")


In [ ]:
print("Rows:", len(df), "| Columns:", df.columns.tolist())
print("Missing :", df["text"].isna().sum(), "| Duplicate text:", df["text"].duplicated().sum())
print("Median words per message:", df["text"].str.split().str.len().median())
print("Labels:", df["label"].value_counts().to_dict())
print("Spam share:", round((df["label"] == "spam").mean() * 100, 1), "%")
print("Example ham: ", df[df["label"] == "ham"]["text"].iloc[0])
print("Example spam:", df[df["label"] == "spam"]["text"].iloc[0])

Rows: 5140 | Columns: ['Unnamed: 0', 'label', 'text']
Missing : 0 | Duplicate text: 0
Median words per message: 12.0
Labels: {'ham': 4498, 'spam': 642}
Spam share: 12.5 %
Example ham:  Watching tv lor. Nice one then i like lor.
Example spam: U are subscribed to the best Mobile Content Service in the UK for å£3 per 10 days until you send STOP to 82324. Helpline 08706091795


 2) train/test split
 70% training, 30% test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=RANDOM_STATE, stratify=df["label"]
)

In [ ]:
print("Training messages:", len(X_train), "| spam share:", round((y_train == "spam").mean() * 100, 1), "%")
print("Test messages:    ", len(X_test), "| spam share:", round((y_test == "spam").mean() * 100, 1), "%")

Training messages: 3598 | spam share: 12.5 %
Test messages:     1542 | spam share: 12.5 %


# 3) FIVE CLASSIFICATION PIPELINES

In [ ]:
# Model 1
model_1 = Pipeline([
    ("count", CountVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("model", MultinomialNB()),
])

model_1.fit(X_train, y_train)
predictions_1 = model_1.predict(X_test)

print(
    "Model 1 | Accuracy:",
    round(accuracy_score(y_test, predictions_1), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_1, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_1, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_1, pos_label="spam"), 3)
)


# Model 2
model_2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("model", MultinomialNB()),
])

model_2.fit(X_train, y_train)
predictions_2 = model_2.predict(X_test)

print(
    "Model 2 | Accuracy:",
    round(accuracy_score(y_test, predictions_2), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_2, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_2, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_2, pos_label="spam"), 3)
)


# Model 3
model_3 = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    )),
])

model_3.fit(X_train, y_train)
predictions_3 = model_3.predict(X_test)

print(
    "Model 3 | Accuracy:",
    round(accuracy_score(y_test, predictions_3), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_3, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_3, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_3, pos_label="spam"), 3)
)


# Model 4
model_4 = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )),
])

model_4.fit(X_train, y_train)
predictions_4 = model_4.predict(X_test)

print(
    "Model 4 | Accuracy:",
    round(accuracy_score(y_test, predictions_4), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_4, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_4, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_4, pos_label="spam"), 3)
)


# Model 5
model_5 = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )),
])

model_5.fit(X_train, y_train)
predictions_5 = model_5.predict(X_test)

print(
    "Model 5 | Accuracy:",
    round(accuracy_score(y_test, predictions_5), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_5, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_5, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_5, pos_label="spam"), 3)
)

Model 1 | Accuracy: 0.977 | Precision: 0.97 | Recall: 0.845 | F1: 0.903
Model 2 | Accuracy: 0.93 | Precision: 1.0 | Recall: 0.44 | F1: 0.612
Model 3 | Accuracy: 0.946 | Precision: 0.991 | Recall: 0.57 | F1: 0.724
Model 4 | Accuracy: 0.977 | Precision: 0.924 | Recall: 0.886 | F1: 0.905
Model 5 | Accuracy: 0.975 | Precision: 0.923 | Recall: 0.87 | F1: 0.896


Model comparison

In [ ]:
# Model Comparison

comparison = pd.DataFrame({
    "Model": [
        "Model 1 - CountVectorizer + NB",
        "Model 2 - TF-IDF + NB",
        "Model 3 - TF-IDF + Logistic Regression",
        "Model 4 - TF-IDF + Balanced Logistic Regression",
        "Model 5 - TF-IDF + Stopwords + Balanced Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, predictions_1),
        accuracy_score(y_test, predictions_2),
        accuracy_score(y_test, predictions_3),
        accuracy_score(y_test, predictions_4),
        accuracy_score(y_test, predictions_5)
    ],
    "Spam Precision": [
        precision_score(y_test, predictions_1, pos_label="spam"),
        precision_score(y_test, predictions_2, pos_label="spam"),
        precision_score(y_test, predictions_3, pos_label="spam"),
        precision_score(y_test, predictions_4, pos_label="spam"),
        precision_score(y_test, predictions_5, pos_label="spam")
    ],
    "Spam Recall": [
        recall_score(y_test, predictions_1, pos_label="spam"),
        recall_score(y_test, predictions_2, pos_label="spam"),
        recall_score(y_test, predictions_3, pos_label="spam"),
        recall_score(y_test, predictions_4, pos_label="spam"),
        recall_score(y_test, predictions_5, pos_label="spam")
    ],
    "Spam F1": [
        f1_score(y_test, predictions_1, pos_label="spam"),
        f1_score(y_test, predictions_2, pos_label="spam"),
        f1_score(y_test, predictions_3, pos_label="spam"),
        f1_score(y_test, predictions_4, pos_label="spam"),
        f1_score(y_test, predictions_5, pos_label="spam")
    ]
})

comparison.round(3)

,Model,Accuracy,Spam Precision,Spam Recall,Spam F1
0,Model 1 - CountVectorizer + NB,0.977,0.970,0.845,0.903
1,Model 2 - TF-IDF + NB,0.930,1.000,0.440,0.612
2,Model 3 - TF-IDF + Logistic Regression,0.946,0.991,0.570,0.724
3,Model 4 - TF-IDF + Balanced Logistic Regression,0.977,0.924,0.886,0.905
4,Model 5 - TF-IDF + Stopwords + Balanced Logist...,0.975,0.923,0.870,0.896


Confusion matrix

In [ ]:
# Confusion Matrix - Model 3

cm_3 = confusion_matrix(
    y_test,
    predictions_3,
    labels=["ham", "spam"]
)

print("Model 3 Confusion Matrix:")
print(cm_3)

tn_3, fp_3, fn_3, tp_3 = cm_3.ravel()

print("TN:", tn_3)
print("FP:", fp_3)
print("FN:", fn_3)
print("TP:", tp_3)

Model 3 Confusion Matrix:
[[1348    1]
 [  83  110]]
TN: 1348
FP: 1
FN: 83
TP: 110


In [ ]:
# Confusion Matrix - Model 4

cm_4 = confusion_matrix(
    y_test,
    predictions_4,
    labels=["ham", "spam"]
)

print("Model 4 Confusion Matrix:")
print(cm_4)

tn_4, fp_4, fn_4, tp_4 = cm_4.ravel()

print("TN:", tn_4)
print("FP:", fp_4)
print("FN:", fn_4)
print("TP:", tp_4)

Model 4 Confusion Matrix:
[[1335   14]
 [  22  171]]
TN: 1335
FP: 14
FN: 22
TP: 171


Task 6 — Hand Calculation

In [ ]:
# Hand Calculation - Model 4

precision_hand = tp_4 / (tp_4 + fp_4)

recall_hand = tp_4 / (tp_4 + fn_4)

f1_hand = 2 * (precision_hand * recall_hand) / (
    precision_hand + recall_hand
)

print("Hand-calculated Precision:", round(precision_hand, 3))
print("Hand-calculated Recall:", round(recall_hand, 3))
print("Hand-calculated F1:", round(f1_hand, 3))

Hand-calculated Precision: 0.924
Hand-calculated Recall: 0.886
Hand-calculated F1: 0.905


In [ ]:
# Model 3 vs Model 4

additional_spam_caught = tp_4 - tp_3
extra_false_alarms = fp_4 - fp_3

print("Additional spam caught:", additional_spam_caught)
print("Extra false alarms:", extra_false_alarms)

Additional spam caught: 61
Extra false alarms: 13


Task 8: Feature Weights

In [ ]:
#Top Feature Weights for Model 4

vectorizer_4 = model_4.named_steps["tfidf"]
classifier_4 = model_4.named_steps["model"]

feature_names = vectorizer_4.get_feature_names_out()
coefficients = classifier_4.coef_[0]

feature_weights = pd.DataFrame({
    "feature": feature_names,
    "weight": coefficients
})

top_positive = feature_weights.sort_values(
    "weight", ascending=False
).head(10)

top_negative = feature_weights.sort_values(
    "weight", ascending=True
).head(10)

print("Top 10 Positive Features:")
print(top_positive)

print("\nTop 10 Negative Features:")
print(top_negative)

Top 10 Positive Features:
      feature    weight
5922     call  5.649560
33273     txt  4.468106
11618    free  4.378961
32107      to  3.972732
30242    text  3.963546
29291    stop  3.503293
37375    your  3.264882
6964    claim  3.145137
20157  mobile  3.052044
23372      or  3.012214

Top 10 Negative Features:
      feature    weight
20823      my -2.438815
19374      me -2.294871
16363      it -2.140741
30435    that -1.726157
22776      ok -1.708764
5598      but -1.523638
7284     come -1.481953
14987     how -1.385083
6203      can -1.377699
18341      ll -1.312271


In [ ]:
# Task 9 - Does Preprocessing Help?

lemmatizer = WordNetLemmatizer()


def clean_letters_only(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(
        lemmatizer.lemmatize(token)
        for token in word_tokenize(text)
    )


def clean_with_markers(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " url ", text)
    text = re.sub(r"\d{5,}", " phonenumber ", text)
    text = re.sub(r"£\s?\d+|\d+p\b", " money ", text)

    return " ".join(
        lemmatizer.lemmatize(token)
        for token in word_tokenize(text)
    )


example = "Call 08712345678 to claim your £1000 prize! www.win.com"

print("\n=== 9) DOES PREPROCESSING HELP? ===")
print("Example message:", example)
print("Letters only:", clean_letters_only(example))
print("Markers:     ", clean_with_markers(example))


# Raw text
model_raw = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model_raw.fit(X_train, y_train)
predictions_raw = model_raw.predict(X_test)


# Letters only
X_train_letters = X_train.apply(clean_letters_only)
X_test_letters = X_test.apply(clean_letters_only)

model_letters = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model_letters.fit(X_train_letters, y_train)
predictions_letters = model_letters.predict(X_test_letters)


# Markers
X_train_markers = X_train.apply(clean_with_markers)
X_test_markers = X_test.apply(clean_with_markers)

model_markers = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model_markers.fit(X_train_markers, y_train)
predictions_markers = model_markers.predict(X_test_markers)


print("\n=== Preprocessing Comparison ===")

print(
    "Raw text     | Accuracy:",
    round(accuracy_score(y_test, predictions_raw), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_raw, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_raw, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_raw, pos_label="spam"), 3)
)

print(
    "Letters only | Accuracy:",
    round(accuracy_score(y_test, predictions_letters), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_letters, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_letters, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_letters, pos_label="spam"), 3)
)

print(
    "Markers      | Accuracy:",
    round(accuracy_score(y_test, predictions_markers), 3),
    "| Precision:",
    round(precision_score(y_test, predictions_markers, pos_label="spam"), 3),
    "| Recall:",
    round(recall_score(y_test, predictions_markers, pos_label="spam"), 3),
    "| F1:",
    round(f1_score(y_test, predictions_markers, pos_label="spam"), 3)
)


=== 9) DOES PREPROCESSING HELP? ===
Example message: Call 08712345678 to claim your £1000 prize! www.win.com
Letters only: call to claim your prize www win com
Markers:      call phonenumber to claim your money prize ! url

=== Preprocessing Comparison ===
Raw text     | Accuracy: 0.975 | Precision: 0.923 | Recall: 0.87 | F1: 0.896
Letters only | Accuracy: 0.971 | Precision: 0.898 | Recall: 0.87 | F1: 0.884
Markers      | Accuracy: 0.981 | Precision: 0.941 | Recall: 0.907 | F1: 0.923


### 9. Does Preprocessing Help?

The preprocessing comparison shows that using meaningful markers improves the classification performance. The raw text model achieved an accuracy of 0.975 and an F1-score of 0.896. The letters-only preprocessing did not improve performance, with an F1-score of 0.884.

The marker-based preprocessing performed best, achieving 0.981 accuracy, 0.941 precision, 0.907 recall, and 0.923 F1-score. Replacing phone numbers, money amounts, and URLs with markers such as `phonenumber`, `money`, and `url` appears to preserve useful spam-related information while making the text easier for the classifier to learn from.

Therefore, the marker-based preprocessing is the best of the three approaches because it achieved the highest accuracy, precision, recall, and F1-score.

In [ ]:
# Model 4: Balanced Logistic Regression

balanced_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )),
])

balanced_model.fit(X_train, y_train)

balanced_predictions = balanced_model.predict(X_test)

print("Balanced predictions created:", len(balanced_predictions))

Balanced predictions created: 1542


In [ ]:
# ============================================
# 10) ERROR ANALYSIS
# ============================================

results = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "predicted": balanced_predictions
})

# Missed spam = actual spam, predicted ham
missed_spam = results[
    (results["true"] == "spam") &
    (results["predicted"] == "ham")
]

# False alarms = actual ham, predicted spam
false_alarms = results[
    (results["true"] == "ham") &
    (results["predicted"] == "spam")
]

print("\n=== 10) ERROR ANALYSIS (Balanced Logistic Regression) ===")

print("Missed spam:", len(missed_spam))
print("False alarms:", len(false_alarms))

print("\n--- 5 MISSED SPAM MESSAGES ---")
for i, text in enumerate(missed_spam["text"].head(5), 1):
    print(f"{i}. {text}")

print("\n--- 5 FALSE ALARM MESSAGES ---")
for i, text in enumerate(false_alarms["text"].head(5), 1):
    print(f"{i}. {text}")


=== 10) ERROR ANALYSIS (Balanced Logistic Regression) ===
Missed spam: 25
False alarms: 14

--- 5 MISSED SPAM MESSAGES ---
1. INTERFLORA - åÒIt's not too late to order Interflora flowers for christmas call 0800 505060 to place your order before Midnight tomorrow.
2. Did you hear about the new \Divorce Barbie\"? It comes with all of Ken's stuff!"
3. How come it takes so little time for a child who is afraid of the dark to become a teenager who wants to stay out all night?
4. Santa Calling! Would your little ones like a call from Santa Xmas eve? Call 09058094583 to book your time.
5. Dorothy@kiefer.com (Bank of Granite issues Strong-Buy) EXPLOSIVE PICK FOR OUR MEMBERS *****UP OVER 300% *********** Nasdaq Symbol CDGT That is a $5.00 per..

--- 5 FALSE ALARM MESSAGES ---
1. Send me the new number
2. On the road so cant txt
3. Text her. If she doesnt reply let me know so i can have her log in
4. Now am free call me pa.
5. Then get some cash together and I'll text jason


### 10. Error Analysis

The balanced Logistic Regression model produced **25 missed spam messages** and **14 false alarms**. The missed spam messages were often promotional or unusual messages, such as Christmas offers, financial promotions, jokes, and messages containing phone numbers. These messages may be difficult to classify because they do not always contain common spam keywords. The false alarms were mostly short and normal personal messages containing words such as “call,” “text,” or “number,” which the model may associate with spam. Overall, the errors show that the model sometimes misses less obvious promotional spam and incorrectly flags short personal messages as spam.

In [ ]:
# ============================================================
# 11) SINGLE-MESSAGE CHECK
# ============================================================

i = 64
one_message = X_test.iloc[i]

print("\n=== 11) SINGLE-MESSAGE CHECK ===")
print("Message             :", one_message)
print("Actual label        :", y_test.iloc[i])

print("Model 3 - Logistic Regression:",
      model_3.predict([one_message])[0])

print("Model 1 - Naive Bayes:",
      model_1.predict([one_message])[0])

print("Model 4 - Balanced Logistic Regression:",
      model_4.predict([one_message])[0])


# ============================================================
# NEW MESSAGES
# ============================================================

new_messages = [
    "URGENT! You have won a free reward, claim now",
    "Are we still meeting for lunch tomorrow?",
    "Call this number to get your prize today",
    "Can you send me the notes from class?",
    "Congratulations! You have won a cash prize. Call now to claim."
]

print("\n=== NEW MESSAGES (Balanced Logistic Regression) ===")

for message in new_messages:
    label = model_4.predict([message])[0]
    spam_probability = model_4.predict_proba([message])[0][1]

    print(
        " ",
        label,
        "| spam probability:",
        round(spam_probability, 2),
        "|",
        message
    )


=== 11) SINGLE-MESSAGE CHECK ===
Message             : Bought one ringtone and now getting texts costing 3 pound offering more tones etc
Actual label        : spam
Model 3 - Logistic Regression: ham
Model 1 - Naive Bayes: spam
Model 4 - Balanced Logistic Regression: ham

=== NEW MESSAGES (Balanced Logistic Regression) ===
  spam | spam probability: 0.96 | URGENT! You have won a free reward, claim now
  ham | spam probability: 0.12 | Are we still meeting for lunch tomorrow?
  spam | spam probability: 0.79 | Call this number to get your prize today
  ham | spam probability: 0.16 | Can you send me the notes from class?
  spam | spam probability: 0.98 | Congratulations! You have won a cash prize. Call now to claim.


### 11. Single-Message Check and New Message Predictions

For the selected test message, the actual label was **spam**. Model 3 (TF-IDF + Logistic Regression) predicted **ham**, while Model 1 (CountVectorizer + Multinomial Naive Bayes) correctly predicted **spam**. Model 4 (TF-IDF + Balanced Logistic Regression) predicted **ham**, so both Logistic Regression models missed this particular spam message.

For the five new messages, the balanced Logistic Regression model correctly identified three clearly promotional messages as spam, with spam probabilities of **0.96, 0.79, and 0.98**. The two normal personal messages were classified as ham, with probabilities of **0.12 and 0.16**. These predictions show that the model can distinguish common promotional spam patterns from ordinary personal communication.